In [8]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [1]:
from huggingface_hub import notebook_login
notebook_login()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [5]:
!pip install transformers  accelerate  datasets
!pip install bitsandbytes
!pip install faiss-cpu

In [6]:
from transformers import AutoTokenizer, AutoModelForCausalLM , BitsAndBytesConfig
import torch

model_name = "meta-llama/Meta-Llama-3-8B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto"
)

config.json:   0%|          | 0.00/654 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/73.0 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/187 [00:00<?, ?B/s]

In [7]:
from sentence_transformers import SentenceTransformer
from datasets import load_dataset

ST = SentenceTransformer('distiluse-base-multilingual-cased-v1')

modules.json:   0%|          | 0.00/341 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/556 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/114 [00:00<?, ?B/s]

2_Dense/model.safetensors:   0%|          | 0.00/1.58M [00:00<?, ?B/s]

In [11]:
folder_file_zip = '/content/drive/MyDrive/TransformesCod/04-hotel/Hotel.zip'
folder = '/content/drive/MyDrive/TransformesCod/04-hotel/'
import zipfile
with zipfile.ZipFile(folder_file_zip, 'r') as zip_ref:
    zip_ref.extractall(folder)

,Question,GoodAnswer
0,أين يقع فندق الفورزد,يقع فندق الفورزد في مدينة بيروت
1,ما هو رقم هاتف فندق الفورزد للاتصال,رقم هاتف فندق الفورزد للاتصال هو 096122334455
2,ما أنواع المطاعم في فندق الفورزد,يوجد في فندق الفورزد مطعم شرقي ومطعم عربي
3,هل يوجد نادي رياضي في فندق الفورزد,يوجد نادي رياضي في فندق الفورزد وصالة جيم
4,هل يوجد مسبح في فندق الفورزد,يوجد مسبح في فندق الفورزد على شاطئ البحر


In [13]:
import pandas as pd
df = pd.read_csv(folder + 'Hotel.csv')
df['text'] = df['Question'] + ' ' + df['GoodAnswer']
df['emmbedding'] = df['text'].apply(lambda x: ST.encode(x))
df[['text' , 'emmbedding']].head(2)

,text,emmbedding
0,أين يقع فندق الفورزد يقع فندق الفورزد في مدينة...,"[0.0053862906, -0.035332885, -0.04001365, -0.0..."
1,ما هو رقم هاتف فندق الفورزد للاتصال رقم هاتف ف...,"[-0.0920274, -0.022635918, -0.05865598, -0.016..."


In [14]:
from datasets import Dataset
dataset = Dataset.from_pandas(df)
dataset

Dataset({
    features: ['Question', 'GoodAnswer', 'text', 'emmbedding'],
    num_rows: 31
})

In [18]:
import faiss
import numpy as np

dataset = dataset.add_faiss_index(column='emmbedding')

  0%|          | 0/1 [00:00<?, ?it/s]

In [19]:
SYS_PROMPT = """You are an assistant for answering questions.
Only give the answer
Dont add any comments or extra explanations.
If you don't know the answer, just say "I do not know." Don't make up an answer.
Answer in Arabic if the question is in Arabic
"""
def search(question , k=2):
  query_embedding = ST.encode(question)
  scores, retrieved_sent = dataset.get_nearest_examples("emmbedding", query_embedding, k=k)
  return scores, retrieved_sent

In [20]:
def format_prompt(question, retrieved_sent , k):
  PROMPT = f"My Question is : {question}\nContext:"
  for ke in range(k):
    PROMPT +=f"{retrieved_sent['text'][ke]}\n"
  return PROMPT

In [39]:
def talk(question):
  k=2
  scores, retrieved_sent = search(question , k)
  formatted_prompt = format_prompt(question , retrieved_sent , k)
  messages = [
      {"role": "system", "content": SYS_PROMPT},
      {"role": "user", "content": formatted_prompt}
  ]

  inputs = tokenizer.apply_chat_template(messages ,return_tensors='pt' , add_generation_prompt = True).to('cuda')
  pad_token_id = tokenizer.eos_token_id if tokenizer.eos_token_id is not None else tokenizer.cos_token_id

  outputs = model.generate(
      inputs['input_ids'],
      attention_mask=inputs['attention_mask'],
      max_new_tokens=50,
      pad_token_id=pad_token_id,
      return_dict_in_generate=True,
      do_sample=True,
      top_p=0.95,
      temperature=0.01,
      repetition_penalty=1.2
  )
  output = tokenizer.decode(outputs.sequences[0] , skip_special_tokens=True)
  return output

In [23]:
def get_last_line(s):
  lines = s.splitlines()
  return lines[-1] if lines else ''

In [40]:
question = "ما أسعار الغرف في فندق الفورزد"
result = talk(question)
last_line = get_last_line(result)
print(last_line)

تتراوح أسعار الغرف بين 100 و 200 دولار. (Between $100 and $200.)
